In [ ]:
## imports module
import anndata as ad
import gc, os
from rich import print
import psutil
import seaborn as sea
import harmonypy as hm

import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy import sparse
from scipy.spatial.distance import pdist
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

In [ ]:
## general setting
sc.settings.verbosity = 0
sc.settings.set_figure_params(dpi=80)
pd.set_option('display.max_columns', None)

# check how many memory used
print(psutil.virtual_memory())
# Force garbage collection
gc.collect()  
sc.settings.set_figure_params(dpi=200, facecolor="white")
PATH = "/home/kibr/Work/Vascular_atlas"

## set path and parameters
os.chdir(PATH)
sc.set_figure_params(figsize=(6, 6), frameon=False)

In [ ]:
## --- Load data --- ##
## Setting one: all endothelial cells
adata = sc.read_h5ad(PATH + "/Results/Revision_2/clean_object_revision_04242026.h5ad")#vnoised',color="cell.class_after_decontX", legend_loc="on data")
# adata = adata[adata.obs['Cell_class'].isin(['Endothelial'])].copy()
## Show the data
print(adata)
print(adata.X[:10,:10])

In [ ]:
adata.obs['Cell_type'].value_counts()

In [ ]:
## create a middle layer of annotation, keep cell class, but split the mural cells into pericytes and smooth muscle cells
adata.obs['Cell_class_middle'] = adata.obs['Cell_class'].astype(str).copy()
adata.obs.loc[adata.obs['Cell_type'] == 'Pericyte', 'Cell_class_middle'] = 'Pericyte'
adata.obs.loc[adata.obs['Cell_type'].isin(['SMC_1','SMC_2', 'SMC_3']), 'Cell_class_middle'] = 'SMC'

adata.obs['Cell_class_middle'].value_counts()

In [ ]:
"""
Regional program strength via single-cell variance partitioning on HVGs.

For each cell type:
  1. Select top N HVGs (donor-aware) so we work in the biologically
     informative subspace and avoid noise-dominated genes.
  2. For each HVG, decompose total cross-cell variance into the part
     explained by region identity (eta^2_region) and the part explained
     by donor identity (eta^2_donor), using a one-way SS decomposition.
  3. Summarize per cell type by averaging eta^2 across HVGs.

Notes / caveats:
  - HVG selection is donor-aware (batch_key=sample) so a single outlier
    donor can't drive HVG choice and inflate donor eta^2 artifactually.
  - Region SS and donor SS are computed marginally; if region and donor
    are unbalanced (typical), they share some variance. The ratio is a
    useful heuristic for ranking cell types, not a formal test.
  - HVGs that vary because of region will of course score high on region
    eta^2 — this is fine for *relative* comparison across cell types but
    means absolute eta^2 values should not be over-interpreted.
"""


# ============================================================
# Variance partitioning
# ============================================================

def variance_partition_hvgs(adata, celltype_col, region_col, sample_col,
                            n_hvgs=3000, min_cells_per_region=20,
                            min_regions=10):
    """
    Per cell type, decompose HVG variance into region vs donor components.
    Returns a DataFrame indexed by cell type.
    """
    results = {}

    for ct in adata.obs[celltype_col].unique():
        sub = adata[adata.obs[celltype_col] == ct].copy()

        # Drop regions with too few cells, then require enough regions
        region_counts = sub.obs[region_col].value_counts()
        valid_regions = region_counts[region_counts >= min_cells_per_region].index
        if len(valid_regions) < min_regions:
            continue
        sub = sub[sub.obs[region_col].isin(valid_regions)].copy()

        # Normalize if it looks like raw counts
        if sub.X.max() > 50:
            sc.pp.normalize_total(sub, target_sum=1e4)
            sc.pp.log1p(sub)

        # Donor-aware HVG selection — prevents single-donor outliers
        # from dominating the gene set
        try:
            sc.pp.highly_variable_genes(sub, n_top_genes=n_hvgs,
                                        batch_key=sample_col, flavor='seurat')
        except Exception:
            sc.pp.highly_variable_genes(sub, n_top_genes=n_hvgs)

        hvg_mask = sub.var['highly_variable'].values
        X_hvg = sub[:, hvg_mask].X
        if sparse.issparse(X_hvg):
            X_hvg = X_hvg.toarray()

        regions = sub.obs[region_col].values
        donors = sub.obs[sample_col].values
        n_total = len(regions)

        # Total sum of squares per gene
        gene_total_var = X_hvg.var(axis=0)
        ss_total = gene_total_var * n_total
        grand_mean = X_hvg.mean(axis=0)

        # Region SS: sum_r n_r * (mean_r - grand_mean)^2  (per gene)
        ss_region = np.zeros(X_hvg.shape[1])
        for r in np.unique(regions):
            m = regions == r
            ss_region += m.sum() * (X_hvg[m].mean(axis=0) - grand_mean) ** 2
        eta_region = np.divide(ss_region, ss_total,
                               out=np.zeros_like(ss_region), where=ss_total > 0)

        # Donor SS: same structure but grouped by donor
        ss_donor = np.zeros(X_hvg.shape[1])
        for d in np.unique(donors):
            m = donors == d
            ss_donor += m.sum() * (X_hvg[m].mean(axis=0) - grand_mean) ** 2
        eta_donor = np.divide(ss_donor, ss_total,
                              out=np.zeros_like(ss_donor), where=ss_total > 0)

        results[ct] = {
            'n_cells': sub.n_obs,
            'n_regions': len(np.unique(regions)),
            'n_donors': len(np.unique(donors)),
            'mean_total_var': float(gene_total_var.mean()),
            'mean_region_eta2': float(eta_region.mean()),
            'mean_donor_eta2': float(eta_donor.mean()),
            'region_vs_donor_ratio': float(eta_region.mean() /
                                           max(eta_donor.mean(), 1e-6)),
        }

    return pd.DataFrame(results).T


# ============================================================
# Figure
# ============================================================

import matplotlib as mpl

def make_variance_figure(df_var, output_path='variance_partition.pdf'):
    """
    Three panels focused on variance partition:
      A. Region eta^2 vs donor eta^2 per cell type (paired bars)
      B. Region/donor ratio — which cell types have region structure
         that exceeds donor confounds
      C. Cell count vs region eta^2 — sanity check that signal isn't
         purely a function of how many cells you have
    """
    # Make text editable in vector outputs (PDF/SVG/EPS)
    mpl.rcParams['pdf.fonttype'] = 42   # TrueType, not Type 3 outlines
    mpl.rcParams['ps.fonttype'] = 42
    mpl.rcParams['svg.fonttype'] = 'none'  # keep text as <text>, not paths
    mpl.rcParams['font.family'] = 'sans-serif'
    mpl.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']

    df = df_var.apply(pd.to_numeric, errors='coerce')
    df = df.dropna(subset=['mean_region_eta2', 'mean_donor_eta2',
                           'region_vs_donor_ratio'])
    df = df.sort_values('mean_region_eta2', ascending=False)

    fig = plt.figure(figsize=(11, 10))
    gs = GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.45,
                  height_ratios=[1.2, 1])

    # ---- Panel A: paired bars, region vs donor eta^2 ----
    ax1 = fig.add_subplot(gs[0, :])
    y = np.arange(len(df))
    h = 0.4

    ax1.barh(y - h/2, df['mean_region_eta2'].values, height=h,
             color='#2c7fb8', edgecolor='black', linewidth=0.4,
             label='Region η²')
    ax1.barh(y + h/2, df['mean_donor_eta2'].values, height=h,
             color='#d95f0e', edgecolor='black', linewidth=0.4,
             label='Donor η²')

    ax1.set_yticks(y)
    ax1.set_yticklabels(df.index, fontsize=9)
    ax1.invert_yaxis()
    ax1.set_xlabel('Mean η² across HVGs (fraction of variance explained)',
                   fontsize=10)
    ax1.set_title('A. Variance attributable to region vs donor, per cell type',
                  fontsize=12, fontweight='bold', loc='left')
    ax1.legend(loc='lower right', fontsize=9, framealpha=0.95)
    ax1.grid(axis='x', alpha=0.3)

    # ---- Panel B: region/donor ratio ----
    ax2 = fig.add_subplot(gs[1, 0])
    df_ratio = df.sort_values('region_vs_donor_ratio', ascending=True)
    y2 = np.arange(len(df_ratio))
    colors = ['#d62728' if r < 1 else '#ff7f0e' if r < 1.5 else '#2ca02c'
              for r in df_ratio['region_vs_donor_ratio'].values]
    ax2.barh(y2, df_ratio['region_vs_donor_ratio'].values,
             color=colors, edgecolor='black', linewidth=0.4)
    ax2.axvline(1, color='black', linestyle='--', linewidth=1, alpha=0.7)
    ax2.set_yticks(y2)
    ax2.set_yticklabels(df_ratio.index, fontsize=8)
    ax2.set_xlabel('Region η² / Donor η²', fontsize=10)
    ax2.set_title('B. Does region beat donor?', fontsize=11,
                  fontweight='bold', loc='left')
    ax2.grid(axis='x', alpha=0.3)

    # ---- Panel C: cell count vs region eta^2 ----
    ax3 = fig.add_subplot(gs[1, 1])
    ax3.scatter(df['n_cells'].values, df['mean_region_eta2'].values,
                c='#2c7fb8', s=80, edgecolors='black', linewidth=0.5)
    ax3.set_xscale('log')
    for ct in df.index:
        ax3.annotate(str(ct),
                     (df.loc[ct, 'n_cells'], df.loc[ct, 'mean_region_eta2']),
                     fontsize=7, xytext=(3, 3), textcoords='offset points')
    ax3.set_xlabel('Cells (log)', fontsize=10)
    ax3.set_ylabel('Region η²', fontsize=10)
    ax3.set_title('C. Signal vs abundance', fontsize=11,
                  fontweight='bold', loc='left')
    ax3.grid(alpha=0.3)

    fig.suptitle('Regional vs donor variance across cell classes (HVG-based)',
                 fontsize=14, fontweight='bold', y=0.995)

    # Save vector format with editable text
    plt.savefig(output_path, bbox_inches='tight')
    # Also save SVG if you prefer that workflow
    plt.savefig(output_path.replace('.pdf', '.svg'), bbox_inches='tight')
    plt.show()
    return df
# ============================================================
# Run
# ============================================================

# if __name__ == '__main__':


#     df_var = variance_partition_hvgs(
#         adata,
#         celltype_col='Cell_class_middle',
#         region_col='region_abb',
#         sample_col='individualID',
#         n_hvgs=3000,
#         min_cells_per_region=20,
#         min_regions=10,
#     )

#     df_var.to_csv('variance_partition.csv')
#     make_variance_figure(df_var, output_path='variance_partition.svg')

In [ ]:
make_variance_figure(df_var, output_path='variance_partition.svg')